In [ ]:
# Key Features:
# - Fast and scalable similarity search for large datasets.
# - Supports both CPU and GPU for high performance.
# - Provides various indexing structures (e.g., flat, IVF, HNSW) for different use cases.

# Real-time Examples:
# 1. Image Search: Given a query image, FAISS can quickly find visually similar images from a large database by comparing feature vectors extracted from images.
# 2. Recommendation Systems: FAISS helps in finding similar users or items based on their embedding vectors, enabling personalized recommendations.
# 3. Natural Language Processing: Used for semantic search, where FAISS retrieves documents or sentences similar to a query based on their vector representations.

# Example Workflow:
# - Extract feature vectors (embeddings) from data (e.g., images, text).
# - Build a FAISS index with these vectors.
# - Perform fast similarity search to find nearest neighbors for a given query vector.

# FAISS is popular in industry and research for powering large-scale search and recommendation engines.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain.embeddings import OllamaEmbeddings
from langchain.text_splitter import CharacterTextSplitter

# Load documents from a text file
loader = TextLoader("speech.txt")
documents = loader.load()
documents



[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness be

In [7]:
# Split documents into smaller chunks for better indexing
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
texts = text_splitter.split_documents(documents)
texts

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [9]:
embedding = OllamaEmbeddings()

db = FAISS.from_documents(texts, embedding)
db


In [12]:
query = "We have borne with their present government through all these bitter months because of that friendship—exercising a patience and forbearance"
results = db.similarity_search(query)
results

[Document(id='187fa6ae-e691-4cd5-8263-d9b741447261', metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(id='a6b28b7d-f91b-47e9-8329-066d874da905', metadata={'sour

In [15]:
# Using Retriever to get the most similar document
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content



'The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'

In [16]:
# With score
results_with_score = db.similarity_search_with_score(query)
results_with_score

[(Document(id='187fa6ae-e691-4cd5-8263-d9b741447261', metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
  np.float32(23837.363)),
 (Document(id='a6b28b7d-f91b-47e9-8329-06

In [23]:
# Save to local storage
db.save_local("faiss_index")

In [26]:
#Load the index from local storage
db_loaded = FAISS.load_local("faiss_index", embedding, allow_dangerous_deserialization=True)
db_loaded
docs = db_loaded.similarity_search(query)
docs[0].page_content

'The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'